# Task 2.4 — Fit the Scaling Law

In [ ]:
import os, json
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), 'ML-Final-Project'))
if not os.path.exists('configs'):
    os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def power_law(N, a, alpha, c):
    return a * N**(-alpha) + c

## Load Training Results

In [ ]:
res_path = 'outputs/results/scaling_results.json'
assert os.path.exists(res_path), f'{res_path} not found — run Task 2.3 first.'

with open(res_path) as f:
    data = json.load(f)

data.sort(key=lambda r: r['param_count'])
names  = [r['name']        for r in data]
params = np.array([r['param_count'] for r in data], dtype=float)
losses = np.array([r['val_loss']    for r in data], dtype=float)

print(f"Loaded {len(data)} models:")
print(f"  {'Name':<10} {'Params':>12} {'Val Loss':>10} {'LR':>8}")
print('  ' + '-'*44)
for r in data:
    print(f"  {r['name']:<10} {r['param_count']:>12,.0f} {r['val_loss']:>10.4f} {r['lr']:>8.0e}")

## Fit the Power Law: $L = a \cdot N^{-\alpha} + c$

In [ ]:
p0 = [1.0, 0.1, max(0.01, losses.min() * 0.5)]
popt, pcov = curve_fit(
    power_law, params, losses,
    p0=p0,
    bounds=([0, 0, 0], [np.inf, 2.0, np.inf]),
    maxfev=20000,
)
a, alpha, c = popt
perr = np.sqrt(np.diag(pcov))

# Goodness of fit
preds = power_law(params, *popt)
ss_res = np.sum((losses - preds)**2)
ss_tot = np.sum((losses - losses.mean())**2)
r2 = 1 - ss_res / ss_tot

print('Fitted parameters:')
print(f'  a     = {a:.4f} ± {perr[0]:.4f}')
print(f'  alpha = {alpha:.4f} ± {perr[1]:.4f}')
print(f'  c     = {c:.4f} ± {perr[2]:.4f}  (irreducible loss floor)')
print(f'  R²    = {r2:.4f}')
print(f'\nL(N) = {a:.4f} · N^(-{alpha:.4f}) + {c:.4f}')
print(f'\nKaplan et al. (NL): alpha ≈ 0.076')
if alpha > 0.076:
    print(f'SVG alpha ({alpha:.3f}) is STEEPER than NL → SVG scales more favorably with model size')
else:
    print(f'SVG alpha ({alpha:.3f}) is SHALLOWER than NL → SVG is harder to improve by scaling alone')

## Log-Log Scaling Plot

In [ ]:
N_fit = np.logspace(np.log10(params.min() * 0.7), np.log10(params.max() * 1.5), 300)
L_fit = power_law(N_fit, *popt)

fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(params, losses, s=100, zorder=5, color='steelblue', label='Trained models')
for n, p, l in zip(names, params, losses):
    ax.annotate(n, (p, l), textcoords='offset points', xytext=(8, 4), fontsize=10)

ax.plot(N_fit, L_fit, '--', color='tomato', linewidth=2,
        label=rf'$L = {a:.3f} \cdot N^{{-{alpha:.3f}}} + {c:.3f}$  ($R^2={r2:.3f}$)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of Parameters (N)', fontsize=13)
ax.set_ylabel('Validation Loss', fontsize=13)
ax.set_title('Task 2.4 — Scaling Law: Val Loss vs. Model Size', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)

# Reference line from Kaplan et al.
# (scaled to match our data range for visual reference)
ax.annotate(f'α = {alpha:.3f} (this work)\nα ≈ 0.076 (Kaplan et al., NL)',
            xy=(0.05, 0.15), xycoords='axes fraction', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

os.makedirs('outputs/plots', exist_ok=True)
fig.tight_layout()
fig.savefig('outputs/plots/scaling_law.png', dpi=150)
plt.show()
print('Saved: outputs/plots/scaling_law.png')

## Task 3.6 Preview — Extrapolation to 880M Parameters

In [ ]:
# Extrapolate to 10× XL (880M params)
xl_params = params.max()
target = xl_params * 10
pred_loss = power_law(target, *popt)

# Uncertainty propagation (delta method)
grad_a     = target**(-alpha)
grad_alpha = -a * np.log(target) * target**(-alpha)
grad_c     = 1.0
pred_std   = np.sqrt((grad_a*perr[0])**2 + (grad_alpha*perr[1])**2 + (grad_c*perr[2])**2)

print(f'Extrapolation: N = {target:.2e} params (10× XL)')
print(f'  Predicted val_loss = {pred_loss:.4f} ± {pred_std:.4f} (1σ)')
print(f'  95% CI: [{pred_loss - 2*pred_std:.4f}, {pred_loss + 2*pred_std:.4f}]')

# Save fit results
os.makedirs('outputs/results', exist_ok=True)
fit_result = {
    'a': a, 'alpha': alpha, 'c': c,
    'a_std': perr[0], 'alpha_std': perr[1], 'c_std': perr[2],
    'r_squared': r2,
    'kaplan_alpha_nl': 0.076,
    'extrapolation': {
        'target_params': target,
        'predicted_val_loss': pred_loss,
        'predicted_val_loss_1sigma': pred_std,
    }
}
with open('outputs/results/scaling_law_fit.json', 'w') as f:
    json.dump(fit_result, f, indent=2)
print('\nFit saved to outputs/results/scaling_law_fit.json')

## Task 2.5 — Training Statistics Table

In [ ]:
print(f"{'Name':<10} {'Params':>10} {'d_model':>8} {'n_lay':>6} {'n_hd':>5} "
      f"{'d_ff':>6} {'ValLoss':>8} {'Time(h)':>8} {'tok/s':>8} {'Mem(MB)':>8}")
print('='*80)
for r in data:
    t = r.get('total_time_seconds', 0)
    print(f"{r['name']:<10} {r['param_count']:>10,} {r['d_model']:>8} "
          f"{r['n_layers']:>6} {r['n_heads']:>5} {r['d_ff']:>6} "
          f"{r['val_loss']:>8.4f} {t/3600:>8.2f} "
          f"{r.get('tokens_per_second',0):>8,.0f} "
          f"{r.get('peak_memory_mb',0):>8.0f}")